# 📊 Sales Performance Analytics and Business Insights
**Dataset:** Sample - Superstore  
**Tools:** Python · Pandas · Matplotlib · Seaborn

---

## 🔹 Section 1 — Data Loading & Preprocessing

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded successfully!')

Libraries loaded successfully!


In [3]:
# Locate CSV (handles both filename variants)
possible_names = [
    'Sample - Superstore.csv.csv',
    'Sample - Superstore.csv',
]
csv_path = None
for name in possible_names:
    if os.path.exists(name):
        csv_path = name
        break
    # Also try alongside the notebook
    nb_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
    candidate = os.path.join(nb_dir, name)
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError('Superstore CSV not found in working directory!')

df = pd.read_csv(csv_path, encoding='latin-1')
print(f'Loaded: {csv_path}')
print(f'Shape: {df.shape}')
df.head()

Loaded: Sample - Superstore.csv.csv
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
# --- Cleaning ---
print(f'Missing values before: {df.isnull().sum().sum()}')
print(f'Duplicate rows before: {df.duplicated().sum()}')

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Convert Order Date
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False, infer_datetime_format=True)

# Extract time features
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Month-Year'] = df['Order Date'].dt.to_period('M')

print(f'\nRecords after cleaning: {df.shape[0]}')
df.dtypes[['Order Date','Year','Month','Month-Year']]

Missing values before: 0
Duplicate rows before: 0


TypeError: to_datetime() got an unexpected keyword argument 'infer_datetime_format'

---
## 🔹 Section 2 — Exploratory Data Analysis (EDA)

In [ ]:
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_profit   = df['Profit'].mean()
margin       = total_profit / total_sales * 100

print(f'Total Sales    : ${total_sales:,.2f}')
print(f'Total Profit   : ${total_profit:,.2f}')
print(f'Average Profit : ${avg_profit:,.2f}')
print(f'Profit Margin  : {margin:.2f}%')

In [ ]:
# Sales & Profit by Category
category_summary = df.groupby('Category')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)
print('--- By Category ---')
display(category_summary)

# Sales & Profit by Region
region_summary = df.groupby('Region')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)
print('\n--- By Region ---')
display(region_summary)

# Sales & Profit by Segment
segment_summary = df.groupby('Segment')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)
print('\n--- By Segment ---')
display(segment_summary)

In [ ]:
# Top 10 Products by Sales
top10 = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10).reset_index()
top10.columns = ['Product Name', 'Total Sales']
top10['Total Sales'] = top10['Total Sales'].map('${:,.2f}'.format)
display(top10)

---
## 🔹 Section 3 — Data Visualization

In [ ]:
# Chart 1: Monthly Sales Trend
monthly_sales = df.groupby('Month-Year')['Sales'].sum().reset_index().sort_values('Month-Year')
monthly_sales['Month-Year-str'] = monthly_sales['Month-Year'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_sales['Month-Year-str'], monthly_sales['Sales'],
        marker='o', linewidth=2, color='#2563EB', markerfacecolor='#EF4444', markersize=5)
ax.fill_between(monthly_sales['Month-Year-str'], monthly_sales['Sales'], alpha=0.12, color='#2563EB')
ax.set_title('Monthly Sales Trend', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month-Year', fontsize=12)
ax.set_ylabel('Total Sales ($)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
step = max(1, len(monthly_sales) // 12)
tick_positions = list(range(0, len(monthly_sales), step))
ax.set_xticks(tick_positions)
ax.set_xticklabels([monthly_sales['Month-Year-str'].iloc[i] for i in tick_positions], rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('monthly_sales_trend.png', dpi=150)
plt.show()
print('Saved: monthly_sales_trend.png')

In [ ]:
# Chart 2: Sales by Region
region_plot = region_summary.reset_index().sort_values('Sales', ascending=False)
colors_region = ['#2563EB', '#10B981', '#F59E0B', '#EF4444']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(region_plot['Region'], region_plot['Sales'], color=colors_region, width=0.55, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Total Sales by Region', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Total Sales ($)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylim(0, region_plot['Sales'].max() * 1.18)
plt.tight_layout()
plt.savefig('sales_by_region.png', dpi=150)
plt.show()
print('Saved: sales_by_region.png')

In [ ]:
# Chart 3: Sales & Profit by Category
cat_plot = category_summary.reset_index()
x     = range(len(cat_plot))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
bars_s = ax.bar([i - width/2 for i in x], cat_plot['Sales'],  width, label='Sales',  color='#2563EB', edgecolor='white')
bars_p = ax.bar([i + width/2 for i in x], cat_plot['Profit'], width, label='Profit', color='#10B981', edgecolor='white')
for bar in bars_s:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8, color='#2563EB')
for bar in bars_p:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8, color='#10B981')
ax.set_title('Sales & Profit by Product Category', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Amount ($)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(list(x))
ax.set_xticklabels(cat_plot['Category'], fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, cat_plot['Sales'].max() * 1.22)
plt.tight_layout()
plt.savefig('sales_profit_by_category.png', dpi=150)
plt.show()
print('Saved: sales_profit_by_category.png')

In [ ]:
# Chart 4: Distribution of Profit
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df['Profit'], bins=50, color='#6366F1', edgecolor='white', alpha=0.85)
ax.axvline(df['Profit'].mean(),   color='#EF4444', linestyle='--', linewidth=1.8, label=f"Mean  ${df['Profit'].mean():,.2f}")
ax.axvline(df['Profit'].median(), color='#F59E0B', linestyle='--', linewidth=1.8, label=f"Median ${df['Profit'].median():,.2f}")
ax.set_title('Distribution of Profit', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Profit ($)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('profit_distribution.png', dpi=150)
plt.show()
print('Saved: profit_distribution.png')

In [ ]:
# Chart 5: Correlation Heatmap
corr_matrix = df[['Sales','Profit','Discount','Quantity']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, linecolor='white', annot_kws={'size': 12},
            ax=ax, vmin=-1, vmax=1, square=True)
ax.set_title('Correlation Heatmap\n(Sales, Profit, Discount, Quantity)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: correlation_heatmap.png')

---
## 🔹 Section 4 — Business Insights

In [ ]:
# INSIGHT 1: Top-performing category by revenue
top_category = category_summary['Sales'].idxmax()
margin_by_cat = (category_summary['Profit'] / category_summary['Sales'] * 100).round(2)
print('💡 Insight 1 — Top Revenue Category')
print(f'   "{top_category}" leads in total sales revenue.')
print(f'   Profit margins: {margin_by_cat.to_dict()}\n')

# INSIGHT 2: Regional performance gap
top_region = region_summary['Sales'].idxmax()
low_region = region_summary['Sales'].idxmin()
print('💡 Insight 2 — Regional Performance Gap')
print(f'   Best region : {top_region}  (${region_summary.loc[top_region,"Sales"]:,.0f})')
print(f'   Worst region: {low_region}  (${region_summary.loc[low_region,"Sales"]:,.0f})')
print(f'   → Focus growth investments on {low_region}\n')

# INSIGHT 3: Furniture has high sales but thin/negative profit margins
furniture_margin = margin_by_cat.get('Furniture', 0)
tech_margin = margin_by_cat.get('Technology', 0)
print('💡 Insight 3 — Furniture: High Sales, Low Profit')
print(f'   Furniture profit margin  : {furniture_margin:.2f}%')
print(f'   Technology profit margin : {tech_margin:.2f}%')
print(f'   → Aggressive discounting in Furniture erodes profits.\n')

# INSIGHT 4: High discount → significantly lower profit (negative correlation)
disc_corr = df[['Discount','Profit']].corr().iloc[0,1]
print('💡 Insight 4 — Discount Hurts Profit')
print(f'   Discount ↔ Profit correlation: {disc_corr:.3f}')
print(f'   → Implement a discount cap policy (max 20%) to protect margins.\n')

# INSIGHT 5: Consumer segment dominates; other segments need attention
seg_share = (segment_summary['Sales'] / segment_summary['Sales'].sum() * 100).round(2)
print('💡 Insight 5 — Segment Revenue Concentration')
for seg, share in seg_share.items():
    print(f'   {seg:<15}: {share:.2f}%')
print(f'   → Invest in Corporate/Home Office to diversify revenue streams.')

print('\n✅ Analysis complete!')